In [44]:
import pandas as pd
import numpy as np

In [573]:
TEST = 'browser_test/browsing_tcp_3.csv'
TEST_OUPUT = 'output/browser/browsing_tcp_3_processed.csv'

# BROWSER_TEST = 'Browsing.csv'
# BROWSER_TEST_OUTPUT = 'output/browser_summary.csv'
# PING_TEST = 'ping_test/ping_tcp_1.csv'
# PING_TEST_OUTPUT = 'output/ping/ping_tcp_1_processed.csv'

In [ ]:
df = pd.read_csv(TEST)

In [575]:
# Get time span before filtering
time_span_raw = df['Time'].max() - df['Time'].min()
print(f"Time span (before processing): {time_span_raw} seconds")

Time span (before processing): 349.2970068 seconds


In [576]:
df.head()

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,0.000000,192.168.2.144,64.233.170.101,TCP,55,54037 > 443 [ACK] Seq=1 Ack=1 Win=509 Len=1
1,2,0.054562,64.233.170.101,192.168.2.144,TCP,66,443 > 54037 [ACK] Seq=1 Ack=2 Win=1045 Len=0...
2,3,0.482534,20.195.65.193,192.168.2.144,TLSv1.2,81,Application Data
3,4,0.526530,192.168.2.144,20.195.65.193,TCP,54,50789 > 443 [ACK] Seq=1 Ack=28 Win=254 Len=0
4,5,0.608588,192.168.2.144,57.144.192.192,TLSv1.2,86,Application Data


In [577]:
df['Protocol'].value_counts()

Protocol
TCP        35646
TLSv1.3    18422
TLSv1.2     5874
SSLv2         51
HTTP          46
Name: count, dtype: int64

In [578]:
# Use this if you want to filter by protocol and port (e.g., HTTP/HTTPS)

# mask_protocol = df['Protocol'].isin(['TCP', 'UDP'])  # Adjust protocol names as needed [e.g., 'TCP', 'UDP']
# mask_port = df['Info'].str.contains(r'\b80\b|\b443\b', na=False)
# df_filtered = df[mask_protocol | mask_port]

df_filtered = df

In [579]:
df_filtered

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,0.000000,192.168.2.144,64.233.170.101,TCP,55,54037 > 443 [ACK] Seq=1 Ack=1 Win=509 Len=1
1,2,0.054562,64.233.170.101,192.168.2.144,TCP,66,443 > 54037 [ACK] Seq=1 Ack=2 Win=1045 Len=0...
2,3,0.482534,20.195.65.193,192.168.2.144,TLSv1.2,81,Application Data
3,4,0.526530,192.168.2.144,20.195.65.193,TCP,54,50789 > 443 [ACK] Seq=1 Ack=28 Win=254 Len=0
4,5,0.608588,192.168.2.144,57.144.192.192,TLSv1.2,86,Application Data
...,...,...,...,...,...,...,...
60034,61452,349.082924,34.225.106.159,192.168.2.144,TLSv1.2,1415,"Certificate, Server Key Exchange, Server Hello..."
60035,61453,349.082987,192.168.2.144,34.225.106.159,TCP,54,50411 > 443 [ACK] Seq=518 Ack=4266 Win=65280...
60036,61454,349.084290,192.168.2.144,34.225.106.159,TLSv1.2,180,"Client Key Exchange, Change Cipher Spec, Encry..."
60037,61455,349.127806,157.240.15.10,192.168.2.144,TCP,60,443 > 53741 [ACK] Seq=7786 Ack=413631 Win=94...


In [580]:
df_filtered.value_counts('Protocol')

Protocol
TCP        35646
TLSv1.3    18422
TLSv1.2     5874
SSLv2         51
HTTP          46
Name: count, dtype: int64

In [581]:
df_filtered.head()

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,0.000000,192.168.2.144,64.233.170.101,TCP,55,54037 > 443 [ACK] Seq=1 Ack=1 Win=509 Len=1
1,2,0.054562,64.233.170.101,192.168.2.144,TCP,66,443 > 54037 [ACK] Seq=1 Ack=2 Win=1045 Len=0...
2,3,0.482534,20.195.65.193,192.168.2.144,TLSv1.2,81,Application Data
3,4,0.526530,192.168.2.144,20.195.65.193,TCP,54,50789 > 443 [ACK] Seq=1 Ack=28 Win=254 Len=0
4,5,0.608588,192.168.2.144,57.144.192.192,TLSv1.2,86,Application Data


In [582]:
df_filtered['Time1'] = df['Time']
df_filtered['Time2'] = df_filtered['Time1'].shift(-1)

df_filtered['Delay'] = df_filtered['Time2'] - df_filtered['Time1']
df_filtered['Delay1'] = df_filtered['Delay'] - df_filtered['Delay'].shift(-1)

df_filtered['Delay2'] = df_filtered['Delay1'].shift(-1)
df_filtered['Jitter'] = np.abs(df_filtered['Delay2'] - df_filtered['Delay1'])

df_filtered.head()

,No.,Time,Source,Destination,Protocol,Length,Info,Time1,Time2,Delay,Delay1,Delay2,Jitter
0,1,0.000000,192.168.2.144,64.233.170.101,TCP,55,54037 > 443 [ACK] Seq=1 Ack=1 Win=509 Len=1,0.000000,0.054562,0.054562,-0.373409,0.383976,0.757385
1,2,0.054562,64.233.170.101,192.168.2.144,TCP,66,443 > 54037 [ACK] Seq=1 Ack=2 Win=1045 Len=0...,0.054562,0.482534,0.427972,0.383976,-0.038062,0.422038
2,3,0.482534,20.195.65.193,192.168.2.144,TLSv1.2,81,Application Data,0.482534,0.526530,0.043996,-0.038062,0.049277,0.087340
3,4,0.526530,192.168.2.144,20.195.65.193,TCP,54,50789 > 443 [ACK] Seq=1 Ack=28 Win=254 Len=0,0.526530,0.608588,0.082058,0.049277,-0.150734,0.200012
4,5,0.608588,192.168.2.144,57.144.192.192,TLSv1.2,86,Application Data,0.608588,0.641369,0.032781,-0.150734,0.153045,0.303779


In [583]:
request_mask = df_filtered['Info'].str.contains(r'^Echo \(ping\) request', case=False)
reply_mask = df_filtered['Info'].str.contains(r'^Echo \(ping\) reply', case=False)

request_df_filtered = df_filtered[request_mask]
reply_df_filtered = df_filtered[reply_mask]

In [584]:
# Reorder columns - No first, then Time1, Time2, etc
cols_order = ['No.', 'Time', 'Time1', 'Time2', 'Delay', 'Delay1', 'Delay2', 'Jitter']
other_cols = [col for col in df_filtered.columns if col not in cols_order]
df_filtered = df_filtered[cols_order + other_cols]
df_filtered.head()

,No.,Time,Time1,Time2,Delay,Delay1,Delay2,Jitter,Source,Destination,Protocol,Length,Info
0,1,0.000000,0.000000,0.054562,0.054562,-0.373409,0.383976,0.757385,192.168.2.144,64.233.170.101,TCP,55,54037 > 443 [ACK] Seq=1 Ack=1 Win=509 Len=1
1,2,0.054562,0.054562,0.482534,0.427972,0.383976,-0.038062,0.422038,64.233.170.101,192.168.2.144,TCP,66,443 > 54037 [ACK] Seq=1 Ack=2 Win=1045 Len=0...
2,3,0.482534,0.482534,0.526530,0.043996,-0.038062,0.049277,0.087340,20.195.65.193,192.168.2.144,TLSv1.2,81,Application Data
3,4,0.526530,0.526530,0.608588,0.082058,0.049277,-0.150734,0.200012,192.168.2.144,20.195.65.193,TCP,54,50789 > 443 [ACK] Seq=1 Ack=28 Win=254 Len=0
4,5,0.608588,0.608588,0.641369,0.032781,-0.150734,0.153045,0.303779,192.168.2.144,57.144.192.192,TLSv1.2,86,Application Data


In [ ]:
df_filtered.to_csv(TEST_OUPUT, index=False)

In [586]:
raise StopIteration("Stop")

StopIteration: Stop

In [606]:
test_1  = pd.read_csv('output/browser/browsing_tcp_1_processed.csv')
test_2  = pd.read_csv('output/browser/browsing_tcp_2_processed.csv')
test_3  = pd.read_csv('output/browser/browsing_tcp_3_processed.csv')

In [607]:
np.sum(test_1['Length'])

np.int64(184855842)

In [608]:
# Load raw data to get raw time spans
raw_test_1 = pd.read_csv('browser_test/browsing_tcp_1.csv')
raw_test_2 = pd.read_csv('browser_test/browsing_tcp_2.csv')
raw_test_3 = pd.read_csv('browser_test/browsing_tcp_3.csv')


# Calculate raw time spans before any filtering
raw_time_span_1 = raw_test_1['Time'].max() - raw_test_1['Time'].min()
raw_time_span_2 = raw_test_2['Time'].max() - raw_test_2['Time'].min()
raw_time_span_3 = raw_test_3['Time'].max() - raw_test_3['Time'].min()

print(f"Raw Time Span Test 1: {raw_time_span_1:.6f} seconds")
print(f"Raw Time Span Test 2: {raw_time_span_2:.6f} seconds")
print(f"Raw Time Span Test 3: {raw_time_span_3:.6f} seconds")

Raw Time Span Test 1: 305.487256 seconds
Raw Time Span Test 2: 306.433924 seconds
Raw Time Span Test 3: 349.297007 seconds


In [609]:
len(raw_test_1)

190493

In [610]:
raw_test_1.head()

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,0.000000,34.104.35.123,192.168.2.144,TCP,1506,"80 > 57506 [PSH, ACK] Seq=1 Ack=1 Win=1050 L..."
1,2,0.000039,192.168.2.144,34.104.35.123,TCP,82,57506 > 80 [ACK] Seq=1 Ack=4294935353 Win=40...
2,3,0.008096,34.104.35.123,192.168.2.144,TCP,1506,80 > 57506 [ACK] Seq=1453 Ack=1 Win=1050 Len...
3,4,0.008133,192.168.2.144,34.104.35.123,TCP,82,[TCP Dup ACK 2#1] 57506 > 80 [ACK] Seq=1 Ack...
4,5,0.011355,34.104.35.123,192.168.2.144,TCP,1506,"80 > 57506 [PSH, ACK] Seq=2905 Ack=1 Win=105..."


In [611]:
raw_test_1['Protocol'].value_counts()

Protocol
TCP        147758
TLSv1.3     40507
TLSv1.2      2036
SSLv2         188
HTTP            2
OCSP            2
Name: count, dtype: int64

In [612]:
test_1['Protocol'].value_counts()

Protocol
TCP        147758
TLSv1.3     40507
TLSv1.2      2036
SSLv2         188
HTTP            2
OCSP            2
Name: count, dtype: int64

In [613]:
throughput_1 = (np.sum(test_1['Length'] * 8) / raw_time_span_1) / 1000
throughput_2 = (np.sum(test_2['Length'] * 8) / raw_time_span_2) / 1000
throughput_3 = (np.sum(test_3['Length'] * 8) / raw_time_span_3) / 1000

print(f"Throughput Test 1: {throughput_1:.6f} kbps")
print(f"Throughput Test 2: {throughput_2:.6f} kbps")
print(f"Throughput Test 3: {throughput_3:.6f} kbps")

Throughput Test 1: 4840.944126 kbps
Throughput Test 2: 7472.071812 kbps
Throughput Test 3: 1264.098299 kbps


In [614]:
# packet_loss_test_1 = pd.read_csv('browser_test/browsing_tcp_lost_segment_1.csv')
# packet_loss_test_2 = pd.read_csv('browser_test/browsing_tcp_lost_segment_2.csv')
# packet_loss_test_3 = pd.read_csv('browser_test/browsing_tcp_lost_segment_3.csv')

In [615]:
# packet_loss_1 = len(raw_test_1) - len(packet_loss_test_1) / len(raw_test_1) * 100
# packet_loss_2 = len(raw_test_2) - len(packet_loss_test_2) / len(raw_test_2) * 100
# packet_loss_3 = len(raw_test_3) - len(packet_loss_test_3) / len(raw_test_3) * 100

In [616]:
total_delay_test_1 = np.sum(test_1['Delay'].dropna())
total_delay_test_2 = np.sum(test_2['Delay'].dropna())
total_delay_test_3 = np.sum(test_3['Delay'].dropna())

In [617]:
print(total_delay_test_1)

305.48725569999635


In [618]:
packet_loss = 0

In [619]:

delay_avg_1 = total_delay_test_1 / len(raw_test_1)
delay_avg_2 = total_delay_test_2 / len(raw_test_2)
delay_avg_3 = total_delay_test_3 / len(raw_test_3)

In [620]:
print(delay_avg_1)

0.0016036665688502797


In [621]:
jitter_1 = np.average(test_1['Jitter'].dropna()) * 1000
jitter_2 = np.average(test_2['Jitter'].dropna()) * 1000
jitter_3 = np.average(test_3['Jitter'].dropna()) * 1000

In [622]:
total_jitter_test_1 = np.sum(test_1['Jitter'].dropna())
total_jitter_test_2 = np.sum(test_2['Jitter'].dropna())
total_jitter_test_3 = np.sum(test_3['Jitter'].dropna())

In [623]:
average_jitter_test_1 = total_jitter_test_1 / len(raw_test_1)
average_jitter_test_2 = total_jitter_test_2 / len(raw_test_2)
average_jitter_test_3 = total_jitter_test_3 / len(raw_test_3)

In [624]:
output_df = pd.DataFrame({
    'Test': ['Test 1', 'Test 2', 'Test 3'],
    'Raw Time Span (s)': [raw_time_span_1, raw_time_span_2, raw_time_span_3],
    'Throughput (kbps)': [throughput_1, throughput_2, throughput_3],
    'Packet Loss (%)': [packet_loss, packet_loss, packet_loss],
    'Total Delay (s)': [total_delay_test_1, total_delay_test_2, total_delay_test_3],
    'Avg Delay (s)': [delay_avg_1, delay_avg_2, delay_avg_3],
    'Total Jitter (s)': [total_jitter_test_1, total_jitter_test_2, total_jitter_test_3],
    'Avg Jitter (s)': [average_jitter_test_1, average_jitter_test_2, average_jitter_test_3],
    'Jitter (ms)': [jitter_1, jitter_2, jitter_3]
})

output_df.to_csv('output/summary.csv', index=False)
print(output_df)

     Test  Raw Time Span (s)  Throughput (kbps)  Packet Loss (%)  \
0  Test 1         305.487256        4840.944126                0   
1  Test 2         306.433924        7472.071812                0   
2  Test 3         349.297007        1264.098299                0   

   Total Delay (s)  Avg Delay (s)  Total Jitter (s)  Avg Jitter (s)  \
0       305.487256       0.001604       1022.552299        0.005368   
1       306.433924       0.001238       1068.922783        0.004318   
2       349.297007       0.005818       1097.865098        0.018286   

   Jitter (ms)  
0     5.368010  
1     4.318444  
2    18.286780  
